# Desicion tree

Загружаем необходимые библиотеки

In [270]:
import numpy as np
import pandas

import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter

from sklearn.model_selection import train_test_split

Выбираем тему seaborn

In [271]:
sns.set(style="whitegrid")

Загружаем датасет

In [272]:
dataset_url = "https://raw.githubusercontent.com/plotly/datasets/refs/heads/master/iris-data.csv"

dataset = pandas.read_csv(dataset_url)

Разбиение датасета на значения

In [273]:
X = dataset.drop(columns=["class"]).values
Y = dataset["class"].values

features = dataset.drop(columns=["class"]).columns.tolist()

Создание объекта Node - decision/leaf 

In [274]:
class Node:
    
    def __init__(self, feature_idx=None, threshold=None, info_gain=None, false_branch=None, true_branch=None, value=None):

        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.false_branch = false_branch
        self.true_branch = true_branch

        self.value = value

Создание объекта дерева

In [275]:
class DecisionTree:

    def __init__(self, min_samples_split=2, max_depth=2):

        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

        self.root = None

    def build_tree(self, data, current_depth=0): # Рекурсия создания дерева
        
        X, Y = data[:, :-1], data[:, -1]
        n_samples, n_features = X.shape
        
        if (n_samples >= self.min_samples_split and current_depth < self.max_depth):
            best_split = self.find_best_split(data, n_features)
            if best_split['information_gain'] > 0:
                false_branch = self.build_tree(best_split['false_data'], current_depth + 1) 
                true_branch = self.build_tree(best_split['true_data'], current_depth + 1)
                return Node(best_split["feature_idx"], best_split["threshold"], best_split["information_gain"], false_branch, true_branch)

        leaf_value = Counter(Y).most_common(1)[0][0]
        return Node(value=leaf_value)
    
    def find_best_split(self, parent_data, n_features): # Поиск наилучшего деления по максимуму изменения information gain

        best_split = {'feature_idx': None, 'threshold': None, 'information_gain': 0, 'false_data': None, 'true_data': None}
        
        for feature_idx in range(n_features):

            thresholds = np.unique(parent_data[:, feature_idx])
            
            for threshold in thresholds:

                false_data, true_data = self.split(parent_data, feature_idx, threshold)
                
                if (len(false_data) and len(true_data)):
                    
                    info_gain = self.information_gain(parent_data, false_data, true_data)
                    
                    if info_gain > best_split['information_gain']:
                        best_split['feature_idx'] = feature_idx
                        best_split['threshold'] = threshold
                        best_split['information_gain'] = info_gain
                        best_split['false_data'] = false_data
                        best_split['true_data'] = true_data

        return best_split           
        
    
    def split(self, parent_data, feature_idx, threshold): # Разделение данных

        false_data = np.array([data for data in parent_data if data[feature_idx] < threshold]) 
        true_data = np.array([data for data in parent_data if data[feature_idx] >= threshold])

        return false_data, true_data


    def entropy(self, data): # Вычисление энтропии

        entropy = 0
        
        class_labels = data[:, -1]
        uniq_class_labels = np.unique(class_labels) # get uniques classes
        
        for class_label in uniq_class_labels:
            p = len(class_labels[class_labels == class_label]) / len(class_labels)
            entropy += -p * np.log2(p) 

        return entropy
        
    def information_gain(self, parent_data, false_data, true_data): # information_gain

        false_p = len(false_data) / len(parent_data)
        true_p = len(true_data) / len(parent_data)
        
        entropy_before_split = self.entropy(parent_data)
        entropy_after_split = false_p * self.entropy(false_data) + true_p * self.entropy(true_data)
        
        return entropy_before_split - entropy_after_split


    def fit(self, X, Y): # Обучение (построение оптимального дерева)
        data = np.concatenate([X, Y.reshape(-1, 1)], axis=1)
        self.root = self.build_tree(data)

    def predict(self, X): # Предсказание на выборке данных
        return np.array([self.predict_class(row, self.root) for row in X])
    
    def predict_one(self, row): # Предсказание на одной выборке данных
        return self.predict_class(row, self.root)
    
    def predict_class(self, data_entry, node): # Предсказание класса с помощью дерева

        if node.value != None:
            return node.value

        feature_value = data_entry[node.feature_idx]

        if feature_value >= node.threshold:
            return self.predict_class(data_entry, node.true_branch)
        else:
            return self.predict_class(data_entry, node.false_branch)

        
    def print_tree(self, features, node=None, depth=0, indent="|   "): # Вывод дерева
        prefix = indent * depth

        if node is None:
            node = self.root

        if node.value is not None:
            print(f"{prefix}|--- class: {node.value}")
            return

        feature_label = f"Feature '{features[node.feature_idx]}'"

        print(f"{prefix}|--- {feature_label} <= {node.threshold}")
        self.print_tree(features, node.false_branch, depth + 1, indent)

        print(f"{prefix}|--- {feature_label} > {node.threshold}")
        self.print_tree(features, node.true_branch, depth + 1, indent)
    

Разбиение датасета на обучающую и проверочную выборку

In [276]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=12)

Конфигурация дерева

In [277]:
tree = DecisionTree(max_depth=1)

Обучение

In [278]:
tree.fit(X_train, Y_train)

tree.print_tree(features)

|--- Feature 'petal length' <= 3.0
|   |--- class: Iris-setosa
|--- Feature 'petal length' > 3.0
|   |--- class: Iris-versicolor


Проверка

In [279]:
correct_predictions = 0
incorrect_predictions = 0

for i in range(len(X_test)):
    if tree.predict_one(X_test[i]) == Y_test[i]:
        correct_predictions += 1
    else:
        incorrect_predictions += 1

print(f" --- Stats ---")
print(f"Correct/Incorrect: {correct_predictions}/{incorrect_predictions}")
print(f"Accuracy: {(correct_predictions * 100 / (incorrect_predictions + correct_predictions)):.2f}% ")

 --- Stats ---
Correct/Incorrect: 21/9
Accuracy: 70.00% 
